# multi_15m_jma_atr_011

> **Auto-generated documentation header.** Fill the `TODO` fields after review.
> Source notebook: `Notebook1 (16).ipynb`

| Field | Value |
|---|---|
| Section | `strategies` |
| Market | `unknown` |
| Trading style | `unknown` |
| Timeframe | `15m` |
| Code cells | 11 |
| Detected functions | resample_1hr_custom, compute_jma_indicator, process_ticker, backtest_trades, calculate_metrics, apply_strategy |

### Research documentation
- **Hypothesis:** _TODO_
- **Alpha source:** _TODO_
- **Indicators / features:** JMA + ATR crossover (detected)
- **Entry logic:** _TODO_
- **Exit logic (SL/TP):** _TODO_
- **Risk management:** _TODO_
- **Assumptions & limitations:** _TODO_
- **Performance (if available):** _TODO_

> Shared utilities live in `src/qresearch/` — prefer importing `jma_signals`, `backtest_trades`,
> `calculate_metrics`, `resample_ohlcv` instead of re-implementing them here.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
data = pd.read_csv('/content/drive/MyDrive/XAU_1m_data.csv')
data.rename(columns={
    'Datetime': 'Date',
    'Open': 'open',
    'Close': 'close',
    'Low': 'low',
    'High': 'high',
    'Volume': 'volume'
}, inplace=True)

data['Ticker'] = 'XAUUSD'

data

In [ ]:
data['Ticker'].unique()

In [ ]:
# Convert 'Date' column to datetime if it isn't already
data['Date'] = pd.to_datetime(data['Date'])

# Define the date range
start_date = '2020-01-01'
end_date = '2025-12-31'

# Define the tickers you want to filter
selected_tickers = ['XAUUSD']  # Add tickers you want

# Apply filtering
data = data[
    (data["Date"] >= start_date) &
    (data["Date"] <= end_date) &
    (data["Ticker"].isin(selected_tickers))
]

# Display filtered results
data

In [ ]:
import pandas as pd

# Ensure Date is datetime
data['Date'] = pd.to_datetime(data['Date'])

# Custom resampling function with offset
def resample_1hr_custom(group):
    ticker = group['Ticker'].iloc[0]
    group = group.set_index('Date')
    resampled = group.resample('15min').agg({
        'open': 'first',
        'high': 'max',
        'low': 'min',
        'close': 'last',
        'volume': 'sum'
    }).dropna().reset_index()
    resampled['Ticker'] = ticker
    return resampled

# Apply per ticker
data_1hr = data.groupby('Ticker').apply(resample_1hr_custom).reset_index(drop=True)

# Reorder columns
data_1hr = data_1hr[['Ticker', 'Date', 'open', 'high', 'low', 'close', 'volume']]

data = data_1hr
data

In [ ]:
data.dropna()
data

In [ ]:
data.columns

In [ ]:
import pandas as pd
import numpy as np
from tqdm import tqdm
from joblib import Parallel, delayed

# === Normalized JMA + Signal in One Function ===
def compute_jma_indicator(
    df,
    smooth_length=10,
    norm_period=90,
    upper_thresh=2,
    lower_thresh=-2.5,
    rs_period=10,
    rs_percentile=10
):
    # === JMA Calculation ===
    price_series = df['close']
    alpha = 2.0 / (smooth_length + 1)
    jma = [price_series.iloc[0]]
    for price in price_series.iloc[1:]:
        jma.append(jma[-1] + (price - jma[-1]) * alpha)
    jma_series = pd.Series(jma, index=price_series.index)

    # === Normalization ===
    mean = jma_series.rolling(window=norm_period).mean()
    std = jma_series.rolling(window=norm_period).std()
    normalized_jma = (jma_series - mean) / std

    # === Rogers-Satchell Volatility ===
    # Replace 0s to avoid log(0)
    open_, high, low, close = df['open'].replace(0, np.nan), df['high'], df['low'], df['close'].replace(0, np.nan)
    log_hc = np.log(high / close)
    log_ho = np.log(high / open_)
    log_lc = np.log(low / close)
    log_lo = np.log(low / open_)
    rs_vol = (log_hc * log_ho + log_lc * log_lo).rolling(window=rs_period).mean()

    # === RS Volatility Filter ===
    rs_thresh = rs_vol.rolling(window=norm_period).quantile(rs_percentile / 100.0)

    # === Generate Signals ===
    long_condition = (normalized_jma > upper_thresh) & (rs_vol > rs_thresh)
    short_condition = (normalized_jma < lower_thresh) & (rs_vol > rs_thresh)
    signal = pd.Series(0, index=df.index)
    signal[long_condition] = 1
    signal[short_condition] = 2

    # === Output ===
    df['jma'] = jma_series
    df['normalized_jma'] = normalized_jma
    df['rs_vol'] = rs_vol
    df['rs_thresh'] = rs_thresh
    df['Signal'] = signal

    return df

# === Per-Ticker Trade Processor ===
def process_ticker(ticker, ticker_data, initial_capital, fee, slippage, tp_pct, sl_pct):
    open_positions = {}
    trades = []
    cumulative_pnl = 0

    for row in ticker_data:
        timestamp, close, open_price, high, low, signal = row["date"], row["close"], row["open"], row["high"], row["low"], row["Signal"]

        if ticker in open_positions:
            position = open_positions[ticker]
            entry_price, direction, qty = position["entry_price"], position["direction"], position["qty"]
            exit_price, exit_reason = None, None

            tp_price = entry_price * (1 + tp_pct) if direction == "long" else entry_price * (1 - tp_pct)
            sl_price = entry_price * (1 - sl_pct) if direction == "long" else entry_price * (1 + sl_pct)

            if (direction == "long" and high >= tp_price) or (direction == "short" and low <= tp_price):
                exit_price = tp_price
                exit_reason = "Take Profit"
            elif (direction == "long" and low <= sl_price) or (direction == "short" and high >= sl_price):
                exit_price = sl_price
                exit_reason = "Stop Loss"

            if exit_price is not None:
                exit_price *= (1 - slippage) if direction == "long" else (1 + slippage)
                raw_pnl = (exit_price - entry_price) * qty if direction == "long" else (entry_price - exit_price) * qty
                pnl = raw_pnl - (fee * entry_price * qty) - (fee * exit_price * qty)
                cumulative_pnl += pnl

                trades.append([ticker, position["entry_time"], timestamp, entry_price, exit_price, pnl, exit_reason, cumulative_pnl, direction])
                del open_positions[ticker]
                continue

        if ticker not in open_positions:
            if signal == 1:
                direction = "long"
            elif signal == 2:
                direction = "short"
            else:
                continue

            entry_price = open_price * (1 + slippage) if direction == "long" else open_price * (1 - slippage)
            qty = initial_capital / entry_price
            open_positions[ticker] = {"entry_time": timestamp, "entry_price": entry_price, "direction": direction, "qty": qty}

    return trades

# === Backtest Function ===
def backtest_trades(data, tp_pct, sl_pct):
    initial_capital = 100000
    fee = 0.0004
    slippage = 0.0005

    unique_tickers = data['Ticker'].unique()

    dtype = [
        ("Ticker", "U10"), ("date", "M8[ms]"), ("close", "f8"),
        ("open", "f8"), ("high", "f8"), ("low", "f8"), ("Signal", "i4")
    ]

    data_np = np.array(
        list(data[["Ticker", "Date", "close", "open", "high", "low", "Signal"]]
             .itertuples(index=False, name=None)),
        dtype=dtype
    )

    results = Parallel(n_jobs=-1)(
        delayed(process_ticker)(
            ticker,
            data_np[data_np['Ticker'] == ticker],
            initial_capital,
            fee,
            slippage,
            tp_pct,
            sl_pct
        ) for ticker in tqdm(unique_tickers)
    )

    trades = [trade for result in results for trade in result]
    return pd.DataFrame(trades, columns=[
        "Ticker", "Entry Time", "Exit Time", "Entry Price", "Exit Price", "PnL", "Exit Reason", "Cumulative PnL", "Direction"
    ])

# === Main Execution ===
# Make sure your 'data' DataFrame exists and has columns: Ticker, Date, close, open, high, low
data['Date'] = pd.to_datetime(data['Date'])

# Compute normalized JMA and generate signals
data = data.groupby('Ticker', group_keys=False).apply(compute_jma_indicator)

# Shift Signal for next-bar execution
data['Signal'] = data.groupby('Ticker')['Signal'].shift(1)
data = data.dropna(subset=['Signal'])
data['Signal'] = data['Signal'].astype(int)

# Run backtest
tradebook = backtest_trades(data, tp_pct=0.1, sl_pct=0.02)

# Save tradebook
tradebook.to_csv("/content/drive/MyDrive/tradebook_Gold.csv", index=False)
tradebook


In [ ]:
from itertools import product
from tqdm import tqdm
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression

# === Metrics Calculation Function (with R² and Monthly Consistency) ===
def calculate_metrics(tradebook):
    if tradebook.empty:
        return 0, 0, 0, 0

    # Ensure datetime
    tradebook['Date'] = pd.to_datetime(tradebook['Entry Time'])  # Replace with correct column if different

    equity = tradebook['PnL'].cumsum()
    total_profit = equity.iloc[-1]

    roll_max = equity.cummax()
    drawdown = roll_max - equity
    max_drawdown = drawdown.max()

    if max_drawdown == 0:
        ratio = float('inf') if total_profit > 0 else 0
    else:
        ratio = total_profit / max_drawdown

    # R² for equity curve smoothness
    x = np.arange(len(equity)).reshape(-1, 1)
    y = equity.values.reshape(-1, 1)
    model = LinearRegression().fit(x, y)
    r2 = model.score(x, y)

    # Monthly return consistency metric
    tradebook['Month'] = tradebook['Date'].dt.to_period('M')
    monthly_returns = tradebook.groupby('Month')['PnL'].sum()
    if len(monthly_returns) < 2:
        monthly_consistency = 0
    else:
        monthly_mean = monthly_returns.mean()
        monthly_std = monthly_returns.std()
        monthly_consistency = monthly_mean / monthly_std if monthly_std > 0 else float('inf')

    return ratio, total_profit, r2, monthly_consistency

# === Hyperparameter Grid ===
param_grid = {
    'smooth_length': [10, 14],
    'norm_period': [100, 120],
    'upper_thresh': [1.75, 2],
    'lower_thresh': [-2, -2.25, -2.5],
    'rs_period': [10, 14],
    'rs_percentile': [10, 30, 50],
    'tp_pct': [0.05, 0.07, 0.1],
    'sl_pct': [0.01, 0.015, 0.02]
}

# === Generate All Parameter Combinations ===
keys, values = zip(*param_grid.items())
param_combinations = list(product(*values))
results = []

print(f"Total combinations to test: {len(param_combinations)}\n")

# === Hyperparameter Tuning Loop ===
for i, combo in enumerate(tqdm(param_combinations, desc="Hypertuning"), 1):
    param = dict(zip(keys, combo))

    print(f"\n=== [{i}/{len(param_combinations)}] Testing Parameters ===")
    for k, v in param.items():
        print(f"{k}: {v}")

    temp_data = data.copy()

    def apply_strategy(group):
        return compute_jma_indicator(
            group,
            smooth_length=param['smooth_length'],
            norm_period=param['norm_period'],
            upper_thresh=param['upper_thresh'],
            lower_thresh=param['lower_thresh'],
            rs_period=param['rs_period'],
            rs_percentile=param['rs_percentile']
        )

    temp_data = temp_data.groupby('Ticker', group_keys=False).apply(apply_strategy)

    # Shift signals to next bar to avoid lookahead bias
    temp_data['Signal'] = temp_data.groupby('Ticker')['Signal'].shift(1)
    temp_data = temp_data.dropna(subset=['Signal'])
    temp_data['Signal'] = temp_data['Signal'].astype(int)

    # Run Backtest
    tradebook = backtest_trades(
        temp_data,
        sl_pct=param['sl_pct'],
        tp_pct=param['tp_pct']
    )

    # Evaluate metrics
    ratio, total_profit, r2, monthly_consistency = calculate_metrics(tradebook)

    print(f"➡️ Profit-to-Drawdown Ratio: {ratio:.4f}")
    print(f"➡️ Total Profit: ₹{total_profit:,.2f}")
    print(f"➡️ Equity Curve R²: {r2:.4f}")
    print(f"➡️ Monthly Consistency Score: {monthly_consistency:.4f}")

    result = {
        **param,
        'Profit/Drawdown': ratio,
        'Total Profit': total_profit,
        'Equity R2': r2,
        'Monthly Consistency': monthly_consistency
    }
    results.append(result)

# === Final Results and Export ===
results_df = pd.DataFrame(results)

# Optional: Composite metric for sorting
results_df['Composite Score'] = (
    results_df['Profit/Drawdown'] * 0.5 +
    results_df['Equity R2'] * 0.4 +
    results_df['Monthly Consistency'] * 0.1
)

results_df = results_df.sort_values(by='Composite Score', ascending=False)
results_df.to_csv("/content/drive/MyDrive/hyperparameter_results_adaptive_intraday_full.csv", index=False)

print("\n✅ Hyperparameter tuning complete. Top 5 results:")
print(results_df.head())
